# 06 — RQ1, tầng 1: ACTIONABLE / NOT_ACTIONABLE

Ở notebook này em huấn luyện và tinh chỉnh nghiêm túc hai chiến lược có huấn luyện cho tầng 1 là ViSoBERT và Bi-LSTM + fastText, rồi
chọn ngưỡng quyết định theo quy tắc đã khóa trong protocol (Q5). LLM few-shot (chiến lược C) được chạy riêng ở Giai đoạn 8 trên máy
local. Trong toàn bộ notebook, Train chỉ dùng để huấn luyện, Validation dùng để chọn hyperparameter và ngưỡng; Hold-out không có mặt
trên Kaggle và chỉ được mở một lần ở Giai đoạn 10.

Em chia công việc thành ba phần. Đầu tiên em dò một lưới hyperparameter nhỏ cho từng mô hình, mỗi cấu hình chạy một seed để tiết kiệm
thời gian, và chọn cấu hình có Macro-F1 cao nhất trên Validation. Sau đó em chạy lại cấu hình thắng với ba seed (42, 1337, 2024). Cuối
cùng em chọn ngưỡng: trong các ngưỡng cho recall của ACTIONABLE ít nhất 0,90 trên Validation, em lấy ngưỡng có Macro-F1 cao nhất. Ngưỡng
được chọn trên lần chạy seed 42 — lần chạy chuẩn theo Q6 — rồi giữ nguyên cho hai seed còn lại, để mỗi mô hình chỉ có đúng một ngưỡng
được khóa lại cho lần đánh giá Hold-out. Vì cùng một Validation được dùng để chọn cấu hình, chọn ngưỡng và báo kết quả, các con số ở
đây sẽ lạc quan hơn thực tế; con số khách quan là kết quả trên Hold-out ở Giai đoạn 10.

Giống notebook 05, em lấy mã nguồn từ repository riêng tư bằng token trong Kaggle Secrets và gỡ token khỏi cấu hình git ngay sau khi
clone.

In [ ]:
import os, subprocess, sys, json
from kaggle_secrets import UserSecretsClient
REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
REPO_DIR = '/kaggle/working/repo'
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
sys.path.append(os.path.join(REPO_DIR, '02_src'))
import yaml, transformers, sentencepiece
from src.utils.seeds import set_seed
set_seed(42)
CFG = yaml.safe_load(open(os.path.join(REPO_DIR, '03_configs', 'frozen', 'classification.yaml'), encoding='utf-8'))
print('transformers', transformers.__version__, '| sentencepiece', sentencepiece.__version__)
print('commit:', subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

## Dữ liệu tầng 1

Nhãn tầng 1 được suy ra từ nhãn 4 lớp: BUG, FEATURE và UX là ACTIONABLE, OTHER là NOT_ACTIONABLE. Ở tầng 1 em giữ cả các review được
đánh dấu ambiguous, vì dù khó phân giữa BUG, FEATURE và UX thì chúng vẫn chắc chắn là actionable.

In [ ]:
import pandas as pd, numpy as np, torch
from src.classification.tier1_actionable.mapping import to_tier1, TIER1_LABELS

def find_data_dir(root='/kaggle/input', target='shopee_train.csv'):
    for dirpath, _, files in os.walk(root):
        if target in files:
            return dirpath
    found = [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs]
    raise FileNotFoundError(f'Không tìm thấy {target} trong {root}. Các file hiện có:\n' + ('\n'.join(found) or '(trống)'))

DATA_DIR = find_data_dir()
train_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_validation.csv'), encoding='utf-8-sig')
T1_TRAIN, T1_VAL = to_tier1(train_df), to_tier1(val_df)
print('Train:', T1_TRAIN['y'].value_counts().rename(dict(enumerate(TIER1_LABELS))).to_dict())
print('Validation:', T1_VAL['y'].value_counts().rename(dict(enumerate(TIER1_LABELS))).to_dict())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có')
from src.utils.kaggle_paths import working_dir
OUT = working_dir('rq1_tier1')
SEEDS = CFG['learning_curve_seeds']

## ViSoBERT: dò hyperparameter

Learning curve ở Giai đoạn 5 dùng cấu hình mặc định và cho recall ACTIONABLE chỉ khoảng 0,58, nên lần này em dò lưới learning rate
{1e-5, 2e-5, 3e-5} và số epoch {3, 4, 5}; các tham số khác giữ như cấu hình đóng băng (max_len 256, batch 16, warmup 10%, weighted
cross-entropy). Mỗi cấu hình chạy một seed, và em chọn theo Macro-F1 trên Validation ở ngưỡng mặc định (lớp có xác suất cao hơn).

In [ ]:
from src.classification.rq1 import (VisobertRunner, BilstmRunner, grid_search, run_seeds, predictions_frame,
                                    cost_row, save_dev_config)
from src.classification.tier1_actionable.threshold import sweep, choose, metrics_at, plot_threshold_curves
from src.classification.learning_curve import tier1_metrics
argmax_metrics = lambda y, probs: tier1_metrics(np.asarray(y), np.asarray(probs).argmax(1))

VIS_GRID = [{'lr': lr, 'epochs': ep} for lr in (1e-5, 2e-5, 3e-5) for ep in (3, 4, 5)]
vis = VisobertRunner(CFG['strategy_b_visobert'])
vis_grid, vis_best = grid_search(vis, VIS_GRID, T1_TRAIN, T1_VAL, 2, TIER1_LABELS, argmax_metrics,
                                 select='macro_f1', seed=42, strategy='visobert')
vis_grid.to_csv(os.path.join(OUT, 'tier1_visobert_grid.csv'), index=False)
print(vis_grid[['config', 'macro_f1', 'f1_actionable', 'recall_actionable', 'precision_actionable', 'train_seconds']].round(4).to_string(index=False))
print('Cấu hình thắng:', vis_best)

## ViSoBERT: ba seed và chọn ngưỡng

In [ ]:
vis_runs = run_seeds(vis, vis_best, SEEDS, T1_TRAIN, T1_VAL, 2, TIER1_LABELS, strategy='visobert')
vis_sweep = sweep(T1_VAL['y'], vis_runs[0]['probs'][:, 1])          # seed 42 = lần chạy chuẩn (Q6)
vis_thr = choose(vis_sweep)
vis_sweep.to_csv(os.path.join(OUT, 'tier1_visobert_threshold_sweep.csv'), index=False)
print('Ngưỡng ViSoBERT:', vis_thr)
predictions_frame(T1_VAL, vis_runs, TIER1_LABELS, 'visobert', 'tier1').to_csv(
    os.path.join(OUT, 'tier1_visobert_val_predictions.csv'), index=False)

## Bi-LSTM + fastText: dò hyperparameter, ba seed và chọn ngưỡng

Với Bi-LSTM em dùng vector fastText tiếng Việt cố định như ở Giai đoạn 5 và dò nhẹ ba tham số: kích thước lớp ẩn {64, 128, 256},
dropout {0,3; 0,5} và learning rate {1e-3; 2e-3}. Quy trình chọn cấu hình, chạy ba seed và chọn ngưỡng giống hệt ViSoBERT để hai mô
hình được so sánh công bằng. Nếu không tải được file vector, em ghi nhận và chỉ báo kết quả ViSoBERT.

In [ ]:
from src.classification.strategy_bilstm_fasttext.embeddings import build_vocab, download, load_vectors
BCFG = CFG['strategy_a_bilstm']
bil_runs = None
try:
    vec = download(BCFG['fasttext_url'], '/tmp/cc.vi.300.vec.gz')
    vocab = build_vocab(pd.concat([train_df['text'], val_df['text']]).astype(str))
    emb, coverage = load_vectors(vec, vocab, dim=int(BCFG['embedding_dim']))
    print(f'Từ vựng {len(vocab):,} | có vector fastText: {coverage:.1%}')
    bil = BilstmRunner(BCFG, vocab, emb)
    BIL_GRID = [{'hidden_size': h, 'dropout': d, 'lr': lr} for h in (64, 128, 256) for d in (0.3, 0.5) for lr in (1e-3, 2e-3)]
    bil_grid, bil_best = grid_search(bil, BIL_GRID, T1_TRAIN, T1_VAL, 2, TIER1_LABELS, argmax_metrics,
                                     select='macro_f1', seed=42, strategy='bilstm')
    bil_grid.to_csv(os.path.join(OUT, 'tier1_bilstm_grid.csv'), index=False)
    print(bil_grid[['config', 'macro_f1', 'f1_actionable', 'recall_actionable', 'train_seconds']].round(4).head(6).to_string(index=False))
    print('Cấu hình thắng:', bil_best)
    bil_runs = run_seeds(bil, bil_best, SEEDS, T1_TRAIN, T1_VAL, 2, TIER1_LABELS, strategy='bilstm')
    bil_sweep = sweep(T1_VAL['y'], bil_runs[0]['probs'][:, 1])
    bil_thr = choose(bil_sweep)
    bil_sweep.to_csv(os.path.join(OUT, 'tier1_bilstm_threshold_sweep.csv'), index=False)
    print('Ngưỡng Bi-LSTM:', bil_thr)
    predictions_frame(T1_VAL, bil_runs, TIER1_LABELS, 'bilstm', 'tier1').to_csv(
        os.path.join(OUT, 'tier1_bilstm_val_predictions.csv'), index=False)
except Exception as e:
    print('Chưa chạy được Bi-LSTM ở notebook này. Lý do:', repr(e))

## Precision và recall theo ngưỡng

Hình dưới cho thấy precision, recall của ACTIONABLE và Macro-F1 thay đổi thế nào khi dịch ngưỡng, đường nét đứt là mức recall 0,90
mà protocol yêu cầu, và đường dọc là ngưỡng em chọn cho từng mô hình.

In [ ]:
sweeps, chosen = {'ViSoBERT': vis_sweep}, {'ViSoBERT': vis_thr}
if bil_runs:
    sweeps['Bi-LSTM + fastText'], chosen['Bi-LSTM + fastText'] = bil_sweep, bil_thr
fig = plot_threshold_curves(sweeps, chosen, os.path.join(OUT, 'tier1_threshold_curves.png'))
from IPython.display import Image, display
display(Image(fig))

## Kết quả tầng 1 trên Validation

Bảng dưới tổng hợp hai mô hình ở cấu hình và ngưỡng đã chọn, trung bình và độ lệch chuẩn qua ba seed, cùng các chỉ số chi phí: thời
gian huấn luyện, thời gian dự đoán cho 1.000 review và bộ nhớ GPU cao nhất. Em cũng lưu cấu hình thắng và ngưỡng vào một file cấu hình
dev; ngưỡng này sẽ được dùng nguyên vẹn khi đánh giá Hold-out ở Giai đoạn 10.

In [ ]:
def summary_rows(runs, thr, strategy, overrides):
    m = pd.DataFrame([metrics_at(T1_VAL['y'], r['probs'][:, 1], thr['threshold']) for r in runs])
    row = {'strategy': strategy, 'cấu hình': json.dumps(overrides), 'ngưỡng': thr['threshold'],
           'đạt recall ≥ 0,90 (seed 42)': thr['recall_constraint_met']}
    for c in ('f1_actionable', 'recall_actionable', 'precision_actionable', 'macro_f1'):
        row[c] = f"{m[c].mean() * 100:.1f} ± {m[c].std(ddof=1) * 100:.1f}"
    return row

perf = [summary_rows(vis_runs, vis_thr, 'ViSoBERT', vis_best)]
costs = [cost_row(vis_runs, 'ViSoBERT')]
if bil_runs:
    perf.append(summary_rows(bil_runs, bil_thr, 'Bi-LSTM + fastText', bil_best))
    costs.append(cost_row(bil_runs, 'Bi-LSTM + fastText'))
perf, costs = pd.DataFrame(perf), pd.DataFrame(costs)
perf.to_csv(os.path.join(OUT, 'tier1_validation_results.csv'), index=False)
costs.to_csv(os.path.join(OUT, 'tier1_costs.csv'), index=False)
print('Hiệu năng trên Validation (%, trung bình ± độ lệch chuẩn, 3 seed, ở ngưỡng đã chọn):')
print(perf.to_string(index=False))
print('\nChi phí:')
print(costs.round(3).to_string(index=False))

dev = {'tier1': {'visobert': {'overrides': vis_best, 'threshold': vis_thr['threshold'],
                              'recall_constraint_met': vis_thr['recall_constraint_met']}}}
if bil_runs:
    dev['tier1']['bilstm'] = {'overrides': bil_best, 'threshold': bil_thr['threshold'],
                              'recall_constraint_met': bil_thr['recall_constraint_met']}
cfg_path = save_dev_config(os.path.join(OUT, 'tier1_dev_config.yaml'), dev,
                           'Tier 1 — cấu hình thắng + ngưỡng chọn trên Validation (seed 42, Q5/Q6). DEV, chưa frozen.\n'
                           'Chép vào 03_configs/dev/ ; đóng băng trước GĐ10.')
print('\nĐã lưu:', sorted(os.listdir(OUT)))

## Nhận xét

Phần nhận xét về hai mô hình em sẽ viết sau khi có kết quả chạy: cấu hình nào thắng và vì sao, ngưỡng chọn được có đạt yêu cầu recall
0,90 hay không và phải đánh đổi bao nhiêu precision, kết quả hai mô hình chênh nhau thế nào so với độ dao động giữa các seed, và chi phí
huấn luyện, dự đoán của mỗi mô hình. Em chưa kết luận mô hình nào tốt hơn ở đây, vì so sánh cuối cùng của RQ1 dựa trên Hold-out và tính
cả chi phí của LLM ở Giai đoạn 8. Các file kết quả trong `/kaggle/working/rq1_tier1/` sẽ được em tải về `05_experiments/rq1_classification/`,
và file `tier1_dev_config.yaml` được chép vào `03_configs/dev/`.